# Setup

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, ConfusionMatrixDisplay
)
 
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 4) 

# Loading Merged EDA Dataset
This is the sa1_merged_eda.csv saved at the end of the EDA notebook.

In [ ]:
DATA_DIR = '../data'  
df = pd.read_csv(os.path.join(DATA_DIR, 'sa1_merged_eda.csv'), parse_dates=['SETTLEMENTDATE'])
 
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range: {df.SETTLEMENTDATE.min()} to {df.SETTLEMENTDATE.max()}')
print(f'Negative price intervals: {(df.RRP < 0).sum():,} ({(df.RRP < 0).mean():.1%})')

# Creating Binary Target Variable
What wer are measuring:
1 = negative price, 0 = zero or positive price.

In [ ]:
df['NEG_PRICE'] = (df['RRP'] < 0).astype(int)
 
print(f'\nTarget distribution:')
print(df['NEG_PRICE'].value_counts())
print(f'\nClass balance: {df.NEG_PRICE.mean():.1%} negative')

 # Feature Engineering
 BLOCK 4 — FEATURE ENGINEERING: LAG FEATURES

 Why lags matter:
   The current RRP is highly correlated with recent RRP values (the ACF
   from the EDA showed this). But using lag_1 (one interval ago) is
   basically cheating — the model is just peeking at near-identical state.

 Strategy (informed by EDA lag correlation decay):
   - Use SPACED lags rather than consecutive ones to reduce multicollinearity
   - lag_1  = 1 interval ago  (30 min)  — strongest signal, but "easy"
   - lag_6  = 6 intervals ago (3 hours) — medium-term memory
   - lag_12 = 12 intervals ago (6 hours) — longer-term context
   - lag_48 = 48 intervals ago (24 hours) — same time yesterday

 For the ablation study, we'll test models with and without short lags
 to show how much performance drops when the model can't "peek".

 IMPORTANT: These lag numbers assume 30-min intervals.
 If you switch to 5-min, multiply by 6 (e.g., lag_6 becomes lag_36).

In [ ]:
lag_features_to_create = {
    'RRP':          [1, 6, 12, 48],   # price memory
    'TOTALDEMAND':  [1, 6, 12],       # demand trends
    'EXCESSGENERATION': [1, 6],       # oversupply signal
    'NETINTERCHANGE':   [1, 6],       # export flow trends
}
 
for col, lags in lag_features_to_create.items():
    for lag in lags:
        lag_col_name = f'{col}_lag{lag}'
        df[lag_col_name] = df[col].shift(lag)
        print(f'  Created: {lag_col_name}')
 
# Drop rows where lag features are NaN (first 48 rows)
rows_before = len(df)
df = df.dropna().reset_index(drop=True)
print(f'\nDropped {rows_before - len(df)} rows with NaN from lag creation')
print(f'Remaining: {len(df):,} rows')
 

# BLOCK 5: DEFINE FEATURE GROUPS (FOR ABLATION STUDY)

 These groups let us test which categories of features matter most.
 The ablation study will train models on each group individually and
 then on all combined, comparing performance.

 Adjust these lists based on your actual EDA correlation rankings.

In [ ]:
demand_features = ['TOTALDEMAND', 'AVAILABLEGENERATION', 'DISPATCHABLEGENERATION',
                   'AVAILABLELOAD', 'DISPATCHABLELOAD', 'DEMANDFORECAST',
                   'INITIALSUPPLY', 'CLEAREDSUPPLY']
 
renewable_features = ['TOTALINTERMITTENTGENERATION', 'UIGF',
                      'SEMISCHEDULE_CLEAREDMW', 'SEMISCHEDULE_COMPLIANCEMW',
                      'CURTAILMENT_MW']
 
interconnector_features = ['NETINTERCHANGE', 'EXCESSGENERATION',
                           'MWFLOW_V-SA', 'MWFLOW_V-SA-MNSP1',
                           'MWLOSSES_V-SA', 'MWLOSSES_V-SA-MNSP1']
 
fcas_features = ['RAISE6SECRRP', 'RAISE60SECRRP', 'RAISE5MINRRP', 'RAISEREGRRP',
                 'LOWER6SECRRP', 'LOWER60SECRRP', 'LOWER5MINRRP', 'LOWERREGRRP']
 
# Lag features
lag_short  = [c for c in df.columns if '_lag1' in c]   # 30-min lookback
lag_medium = [c for c in df.columns if '_lag6' in c or '_lag12' in c]  # 3–6 hour
lag_long   = [c for c in df.columns if '_lag48' in c]  # 24-hour
 
all_lag_features = lag_short + lag_medium + lag_long
 
# Combined: everything
all_features = (demand_features + renewable_features +
                interconnector_features + fcas_features + all_lag_features)
 
# Verify all features exist in the dataframe
missing = [f for f in all_features if f not in df.columns]
if missing:
    print(f'WARNING — features not found in dataset: {missing}')
    all_features = [f for f in all_features if f in df.columns]
else:
    print(f'All {len(all_features)} features verified in dataset')
 